## 1. Import Library

In [ ]:
import os
import subprocess
import pathlib
import tempfile
import numpy as np
import librosa
import soundfile as sf
import imageio_ffmpeg
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Path ke binary ffmpeg dari imageio-ffmpeg (tanpa perlu install ffmpeg di sistem)
FFMPEG_BIN = imageio_ffmpeg.get_ffmpeg_exe()

✅ Semua library berhasil diimport!
🎬 FFmpeg binary: c:\S2-Ilmu Komputer 24\Project\Kambing\.venv\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe


## 2. Konfigurasi Path & Parameter

In [2]:
ROOT = pathlib.Path(r"c:\S2-Ilmu Komputer 24\Project\Kambing")

RAW_DIR = ROOT / "Rawdataset"

# Input folders (berisi file .m4a)
SRC_LAPAR  = RAW_DIR / "lapar"
SRC_NORMAL = RAW_DIR / "normal"

# Output folders (akan berisi file .wav 2 detik)
DST_LAPAR  = RAW_DIR / "lapar_wav"
DST_NORMAL = RAW_DIR / "normal_wav"

DST_LAPAR.mkdir(exist_ok=True)
DST_NORMAL.mkdir(exist_ok=True)

# ─── Parameter audio ───
SEGMENT_DURATION = 2.0     # detik
TARGET_SR        = 22050   # sample rate output WAV

# ─── Parameter deteksi suara kambing ───
SILENCE_THRESHOLD_RMS = 0.01  # RMS minimal agar dianggap ada suara
ZCR_MIN = 0.01                # Di bawah ini → hening
ZCR_MAX = 0.35                # Di atas ini  → noise (gesekan, angin)
SPECTRAL_CENTROID_MAX = 6000  # Hz, di atas ini → noise wideband

print(f"📁 Source lapar  : {SRC_LAPAR}")
print(f"📁 Source normal : {SRC_NORMAL}")
print(f"📁 Output lapar  : {DST_LAPAR}")
print(f"📁 Output normal : {DST_NORMAL}")
print(f"⏱️  Durasi segmen : {SEGMENT_DURATION} detik")
print(f"🎵 Sample rate   : {TARGET_SR} Hz")

📁 Source lapar  : c:\S2-Ilmu Komputer 24\Project\Kambing\Rawdataset\lapar
📁 Source normal : c:\S2-Ilmu Komputer 24\Project\Kambing\Rawdataset\normal
📁 Output lapar  : c:\S2-Ilmu Komputer 24\Project\Kambing\Rawdataset\lapar_wav
📁 Output normal : c:\S2-Ilmu Komputer 24\Project\Kambing\Rawdataset\normal_wav
⏱️  Durasi segmen : 2.0 detik
🎵 Sample rate   : 22050 Hz


## 3. Fungsi Konversi M4A → WAV (via ffmpeg subprocess)

Langkah ini menggunakan binary ffmpeg langsung via `subprocess.run()`,  
sehingga **tidak bergantung pada pydub** dan tidak butuh ffprobe.

In [ ]:
def convert_m4a_to_wav(m4a_path: pathlib.Path, wav_path: pathlib.Path) -> bool:
   
    cmd = [
        FFMPEG_BIN,
        '-y',                    # overwrite jika sudah ada
        '-i', str(m4a_path),     # input file
        '-ac', '1',              # mono
        '-ar', str(TARGET_SR),   # sample rate
        '-sample_fmt', 's16',    # 16-bit PCM
        '-f', 'wav',             # format output
        str(wav_path)            # output file
    ]
    
    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    
    return result.returncode == 0

✅ Fungsi convert_m4a_to_wav siap!


## 4. Fungsi Deteksi Suara Kambing vs Noise/Silence

In [ ]:
def classify_segment(audio_array: np.ndarray, sr: int) -> str:
   
    if audio_array.ndim > 1:
        audio_array = np.mean(audio_array, axis=0)

    # 1. RMS Energy — terlalu pelan = hening
    rms = np.sqrt(np.mean(audio_array ** 2))
    if rms < SILENCE_THRESHOLD_RMS:
        return 'noise'

    # 2. Zero-Crossing Rate — terlalu rendah/tinggi = bukan suara kambing
    zcr = np.mean(librosa.feature.zero_crossing_rate(audio_array))
    if zcr < ZCR_MIN or zcr > ZCR_MAX:
        return 'noise'

    # 3. Spectral Centroid — frekuensi dominan terlalu tinggi = noise wideband
    spectral_centroid = np.mean(
        librosa.feature.spectral_centroid(y=audio_array, sr=sr)
    )
    if spectral_centroid > SPECTRAL_CENTROID_MAX:
        return 'noise'

    # 4. Spectral Flatness — terlalu flat = noise (bukan tonal/biologis)
    spectral_flatness = np.mean(
        librosa.feature.spectral_flatness(y=audio_array)
    )
    if spectral_flatness > 0.5:
        return 'noise'

    return 'goat'

✅ Fungsi classify_segment siap!


## 5. Fungsi Potong WAV → Segmen 2 Detik + Klasifikasi

In [ ]:
def segment_and_classify(wav_path: pathlib.Path) -> tuple:
   
    # Baca wav dengan librosa (sudah mono & TARGET_SR dari konversi ffmpeg)
    y, sr = librosa.load(str(wav_path), sr=TARGET_SR, mono=True)
    
    samples_per_segment = int(SEGMENT_DURATION * sr)  # 2 * 22050 = 44100
    n_segments = len(y) // samples_per_segment
    
    goat_segs  = []
    noise_segs = []
    
    for i in range(n_segments):
        start = i * samples_per_segment
        end   = start + samples_per_segment
        segment = y[start:end]
        
        label = classify_segment(segment, sr)
        
        if label == 'goat':
            goat_segs.append(segment)
        else:
            noise_segs.append(segment)
    
    return goat_segs, noise_segs

✅ Fungsi segment_and_classify siap!


## 6. Fungsi Simpan WAV

In [ ]:
def save_wav(audio_array: np.ndarray,
             output_path: pathlib.Path,
             sr: int = TARGET_SR):
    
    sf.write(str(output_path), audio_array, sr, subtype='PCM_16')

✅ Fungsi save_wav siap!


## 7. Proses Semua File M4A dalam Satu Kelas

**Alur per file:**
1. `m4a` → konversi ke `wav` sementara (via ffmpeg)
2. `wav` sementara → potong 2 detik → klasifikasi
3. Simpan segmen kambing & noise ke folder output

In [ ]:
def process_class(src_folder: pathlib.Path,
                  dst_folder: pathlib.Path,
                  class_label: str):
    
    m4a_files = sorted(src_folder.glob("*.m4a"))
    if not m4a_files:
        print(f"⚠️  Tidak ada file .m4a di {src_folder}")
        return 0, 0

    print(f"\n{'='*60}")
    print(f"📂 Kelas   : {class_label.upper()}")
    print(f"📁 Sumber  : {src_folder}")
    print(f"📁 Output  : {dst_folder}")
    print(f"   ├─ Suara kambing → {class_label}_1.wav, {class_label}_2.wav, ...")
    print(f"   └─ Noise/hening  → noise_1.wav, noise_2.wav, ...")
    print(f"🔢 File M4A: {len(m4a_files)}")
    print(f"{'='*60}")

    goat_counter  = 1
    noise_counter = 1

    existing_goat  = sorted(dst_folder.glob(f"{class_label}_*.wav"))
    existing_noise = sorted(dst_folder.glob("noise_*.wav"))
    if existing_goat:
        last_num = max(int(p.stem.split('_')[-1]) for p in existing_goat)
        goat_counter = last_num + 1
    if existing_noise:
        last_num = max(int(p.stem.split('_')[-1]) for p in existing_noise)
        noise_counter = last_num + 1

    total_goat  = 0
    total_noise = 0

    for m4a_file in tqdm(m4a_files, desc=f"Memproses {class_label}"):
        try:
            # ── STEP 1: Konversi m4a → wav sementara ──────────────
            temp_wav = dst_folder / f"_temp_{m4a_file.stem}.wav"
            
            success = convert_m4a_to_wav(m4a_file, temp_wav)
            if not success:
                print(f"  ✘ {m4a_file.name} — gagal konversi m4a→wav")
                continue
            
            # ── STEP 2: Potong 2 detik + klasifikasi ──────────────
            goat_segs, noise_segs = segment_and_classify(temp_wav)
            
            for seg in goat_segs:
                out_name = f"{class_label}_{goat_counter}.wav"
                save_wav(seg, dst_folder / out_name)
                goat_counter += 1
                total_goat   += 1

            for seg in noise_segs:
                out_name = f"noise_{noise_counter}.wav"
                save_wav(seg, dst_folder / out_name)
                noise_counter += 1
                total_noise   += 1
            
            if temp_wav.exists():
                temp_wav.unlink()

            print(f"  ✔ {m4a_file.name:30s} → "
                  f"{len(goat_segs):3d} kambing, "
                  f"{len(noise_segs):3d} noise")

        except Exception as e:
            print(f"  ✘ {m4a_file.name} — ERROR: {e}")
            temp_wav = dst_folder / f"_temp_{m4a_file.stem}.wav"
            if temp_wav.exists():
                temp_wav.unlink()

    print(f"\n📊 Total {class_label}: {total_goat} kambing | {total_noise} noise")
    return total_goat, total_noise

✅ Fungsi process_class siap!


## 8. Jalankan Proses untuk Kelas Lapar

In [8]:
lapar_goat, lapar_noise = process_class(
    src_folder  = SRC_LAPAR,
    dst_folder  = DST_LAPAR,
    class_label = "lapar"
)


📂 Kelas   : LAPAR
📁 Sumber  : c:\S2-Ilmu Komputer 24\Project\Kambing\Rawdataset\lapar
📁 Output  : c:\S2-Ilmu Komputer 24\Project\Kambing\Rawdataset\lapar_wav
   ├─ Suara kambing → lapar_1.wav, lapar_2.wav, ...
   └─ Noise/hening  → noise_1.wav, noise_2.wav, ...
🔢 File M4A: 17


Memproses lapar:   6%|▌         | 1/17 [00:11<03:10, 11.88s/it]

  ✔ lapar 1.m4a                    → 104 kambing,  63 noise


Memproses lapar:  12%|█▏        | 2/17 [00:13<01:25,  5.67s/it]

  ✔ lapar 2.m4a                    →  77 kambing,  69 noise


...

  ✔ lapar16.m4a                    →  44 kambing,   1 noise


Memproses lapar:  94%|█████████▍| 16/17 [00:41<00:01,  1.81s/it]

  ✔ lapar17.m4a                    → 162 kambing,   2 noise


Memproses lapar: 100%|██████████| 17/17 [00:42<00:00,  2.47s/it]

  ✔ lapar9.m4a                     →  21 kambing,   0 noise

📊 Total lapar: 2602 kambing | 461 noise


## 9. Jalankan Proses untuk Kelas Normal

In [9]:
normal_goat, normal_noise = process_class(
    src_folder  = SRC_NORMAL,
    dst_folder  = DST_NORMAL,
    class_label = "normal"
)


📂 Kelas   : NORMAL
📁 Sumber  : c:\S2-Ilmu Komputer 24\Project\Kambing\Rawdataset\normal
📁 Output  : c:\S2-Ilmu Komputer 24\Project\Kambing\Rawdataset\normal_wav
   ├─ Suara kambing → normal_1.wav, normal_2.wav, ...
   └─ Noise/hening  → noise_1.wav, noise_2.wav, ...
🔢 File M4A: 15


Memproses normal:   7%|▋         | 1/15 [00:01<00:14,  1.01s/it]

  ✔ normal 1.m4a                   →  49 kambing,  75 noise


Memproses normal:  13%|█▎        | 2/15 [00:03<00:21,  1.69s/it]

  ✔ normal 10.m4a                  →  83 kambing,  89 noise


...

  ✔ normal13.m4a                   →  44 kambing,  15 noise


Memproses normal:  93%|█████████▎| 14/15 [00:20<00:01,  1.38s/it]

  ✔ normal14.m4a                   →  74 kambing,  73 noise


Memproses normal: 100%|██████████| 15/15 [00:22<00:00,  1.49s/it]

  ✔ normal15.m4a                   → 112 kambing, 164 noise

📊 Total normal: 1230 kambing | 1109 noise


## 10. Ringkasan Akhir

In [ ]:
print("\n" + "="*65)
print("📋 RINGKASAN KONVERSI")
print("="*65)
print(f"{'Kelas':<12} {'Folder Output':<20} {'Suara Kambing':>14} {'Noise':>8}")
print("-"*58)
print(f"{'lapar':<12} {'lapar_wav/':<20} {lapar_goat:>14} {lapar_noise:>8}")
print(f"{'normal':<12} {'normal_wav/':<20} {normal_goat:>14} {normal_noise:>8}")
print("-"*58)
print(f"{'TOTAL':<12} {'':<20} {lapar_goat + normal_goat:>14} {lapar_noise + normal_noise:>8}")
print("="*65)


📋 RINGKASAN KONVERSI
Kelas        Folder Output         Suara Kambing    Noise
----------------------------------------------------------
lapar        lapar_wav/                     2602      461
normal       normal_wav/                    1230     1109
----------------------------------------------------------
TOTAL                                       3832     1570

📁 Isi folder lapar_wav  : c:\S2-Ilmu Komputer 24\Project\Kambing\Rawdataset\lapar_wav
   ├─ lapar_N.wav  : 2602 file
   └─ noise_N.wav  : 461 file

📁 Isi folder normal_wav : c:\S2-Ilmu Komputer 24\Project\Kambing\Rawdataset\normal_wav
   ├─ normal_N.wav : 1230 file
   └─ noise_N.wav  : 1109 file
